In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

dir_path = '/content/drive/MyDrive/02740_data/homework4' # Path to the folder
os.chdir(dir_path)

In [ ]:
!pip install git+https://github.com/ChristophReich1996/Yeast-in-Microstructures-Dataset.git --no-deps

In [ ]:
!pip install git+https://github.com/Lightning-AI/metrics.git

In [ ]:
!pip install kornia

# Exploring the dataset


Paper link: [An Instance Segmentation Dataset of Yeast Cells in Microstructures.](https://arxiv.org/abs/2304.07597)

The dataset consists of 493 densely annotated microscopy images, each containing pixel-wise instance segmentation labels for both yeast cells and trap microstructures. This level of detailed annotation is crucial for developing and benchmarking segmentation algorithms that can handle the complexities of microscopy imagery, especially in microstructured environments.

Key features of the dataset include:

*   **Pixel-wise instance segmentation labels**: Each image is annotated at the pixel level, providing precise instance-wise segmentation for yeast cells and the microstructures they are trapped in.
*   **Dense annotations**: The dataset contains detailed annotations for each image, ensuring that all relevant structures are accurately labeled.

*   **Standardized evaluation strategy**: To facilitate a fair comparison between different segmentation algorithms, a standardized evaluation strategy is proposed. This ensures that results are comparable and that improvements in segmentation techniques can be accurately measured.

In [ ]:
import yim_dataset
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch
from torch.optim import Adam
import kornia.augmentation as K
import matplotlib.pyplot as plt
import copy

In [ ]:
augmentations = K.AugmentationSequential(
    K.RandomHorizontalFlip(p=0.5),
    K.RandomVerticalFlip(p=0.5),
    K.RandomGaussianBlur(kernel_size=(7,7), sigma=(9, 9), p=0.1),
    K.Normalize(mean=torch.tensor([0.485]), std=torch.tensor([0.229])),
    data_keys=["input", "bbox_xyxy", "mask"],
    same_on_batch=False,
)

# Initialize dataset
train_dataset = yim_dataset.data.YIMDataset(
    path='./yeast_cell_in_microstructures_dataset/train/',
    augmentations=augmentations,
    return_absolute_bounding_box=False
)

# Creating a train loader
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    collate_fn=yim_dataset.data.collate_function_yim_dataset
)

# Validation dataset to evaluate performance after each epoch
val_dataset = yim_dataset.data.YIMDataset(
    path='./yeast_cell_in_microstructures_dataset/val/',
    augmentations=augmentations,
    return_absolute_bounding_box=False
)

# Creating a val loader
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=1,
    collate_fn=yim_dataset.data.collate_function_yim_dataset
)

# Test dataset
test_dataset = yim_dataset.data.YIMDataset(
    path='./yeast_cell_in_microstructures_dataset/test/',
    return_absolute_bounding_box=False
)

# Creating a test loader
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=1,
    collate_fn=yim_dataset.data.collate_function_yim_dataset
)


In [ ]:
# Visualizing the training images with their corresponding mask.
# The image will be our input to the model. The mask will be our
# target.

def visualize_data(train_loader):
    for image, mask, bbox, cls in train_loader:
        combined_Ys = []
        for y in mask:
            combined_Ys.append(torch.max(y, dim=0)[0].unsqueeze(0))

        combined_Y = torch.stack(combined_Ys)
        fig, ax = plt.subplots(1, 2)
        ax[0].imshow(image[0].squeeze().cpu().numpy(), cmap='gray')
        ax[0].set_title('Input Image')
        ax[1].imshow(combined_Y[0].squeeze().cpu().numpy(), cmap='gray')
        ax[1].set_title('Mask')
        plt.show()
        break

visualize_data(train_loader)

# Model Definition

**Model Architecture**

For this tutorial, we will use a U-Net architecture, which is a popular choice for biomedical image segmentation tasks. The U-Net model is designed to work with fewer training images and to provide more precise segmentations.



Here's an overview of the U-Net architecture:

* **Encoder (Contracting Path)**: Consists of a series of convolutional layers followed by max pooling. The purpose is to capture the context of the image through a down-sampling process.

* **Bottleneck**: Connects the encoder and decoder.
Comprises convolutional layers without any down-sampling, preserving the spatial dimensions while increasing the depth of the feature maps.

* **Decoder (Expansive Path)**: Consists of up-sampling layers followed by convolutional layers. The purpose is to reconstruct the spatial dimensions of the image while combining the high-level features from the encoder.

* **Skip Connections**: Connect corresponding layers from the encoder to the decoder and helps in retaining high-resolution features lost during down-sampling.

In [ ]:
class UNet(nn.Module):
    def __init__(self, input_shape = (1, 256, 256), kernel_size = 3):
        super(UNet, self).__init__()

        self.k = kernel_size

        self.enc1 =  self.ConvBlock(1, 64)
        self.enc2 =  self.ConvBlock(64, 128)
        self.enc3 =  self.ConvBlock(128, 256)
        self.enc4 =  self.ConvBlock(256, 512)

        self.pool = nn.MaxPool2d(kernel_size=2)

        self.bottleneck = nn.Sequential(
            nn.Conv2d(512, 1024, self.k, padding = 1),
            nn.BatchNorm2d(1024),
            nn.ReLU(inplace = True),
            nn.Conv2d(1024, 1024, self.k, padding = 1),
            nn.BatchNorm2d(1024),
            nn.ReLU(inplace = True)
        )


        self.upconv4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 =   self.ConvBlock(1024, 512)

        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 =   self.ConvBlock(512, 256)

        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 =   self.ConvBlock(256, 128)

        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 =   self.ConvBlock(128, 64)


        self.final_conv = nn.Conv2d(64, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def ConvBlock(self, in_channels, out_channels):

        _module = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, self.k, padding = 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace = True),
            nn.Conv2d(out_channels, out_channels, self.k, padding = 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace = True),
        )

        return _module


    def forward(self, x):
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool(enc1))
        enc3 = self.enc3(self.pool(enc2))
        enc4 = self.enc4(self.pool(enc3))

        bottleneck_out = self.bottleneck(self.pool(enc4))

        upsample4 = self.upconv4(bottleneck_out)
        upsample4 = torch.cat([upsample4, enc4], dim = 1)
        dec4 = self.dec4(upsample4)

        upsample3 = self.upconv3(dec4)
        upsample3 = torch.cat([upsample3, enc3], dim = 1)
        dec3 = self.dec3(upsample3)

        upsample2 = self.upconv2(dec3)
        upsample2 = torch.cat([upsample2, enc2], dim = 1)
        dec2 = self.dec2(upsample2)

        upsample1 = self.upconv1(dec2)
        upsample1 = torch.cat([upsample1, enc1], dim = 1)
        dec1 = self.dec1(upsample1)



        final_out = self.final_conv(dec1)
        final_out = self.sigmoid(final_out)

        return final_out



In [ ]:
model_1 = UNet()

def weights_init(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.ConvTranspose2d):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

model_1.apply(weights_init)
model_2 = copy.deepcopy(model_1)


### Utilize Existing UNet Architectures

#### Using the segmentation_models_pytorch Library
The segmentation_models_pytorch library offers a collection of pre-trained models for semantic segmentation, including U-Net with various encoder backbones.

In [ ]:
!pip install segmentation-models-pytorch

In [ ]:
import segmentation_models_pytorch as smp

# Create a U-Net model with a specific encoder
model = smp.Unet(
    encoder_name='resnet34',        # Choose encoder, e.g., 'mobilenet_v2', 'resnet50'
    encoder_weights='imagenet',     # Use 'imagenet' pre-trained weights
    in_channels=1,                  # Model input channels (1 for grayscale, 3 for RGB)
    classes=2,                      # Model output channels (number of classes)
)



#### Using the MONAI (Medical Open Network for AI) Library
MONAI is a PyTorch-based framework tailored for healthcare imaging that includes U-Net implementations optimized for medical image segmentation.



In [ ]:
!pip install monai

In [ ]:
from monai.networks.nets import UNet

# Create a U-Net model
model = UNet(
    spatial_dims=2,              # Set to 2 for 2D data or 3 for 3D data
    in_channels=1,               # Number of input channels
    out_channels=2,              # Number of output channels (classes)
    channels=(16, 32, 64, 128, 256),  # Number of features at each layer
    strides=(2, 2, 2, 2),        # Strides for each downsampling layer
    num_res_units=2,             # Number of residual units
)


#### Using the PyTorch Hub
PyTorch Hub is a pre-trained model repository designed for research exploration.

We can load a pre-trained U-Net model for abnormality segmentation on a dataset of brain MRI volumes: kaggle.com/mateuszbuda/lgg-mri-segmentation

In [ ]:
import torch
model = torch.hub.load('mateuszbuda/brain-segmentation-pytorch', 'unet',
    in_channels=1, out_channels=1, init_features=32, pretrained=False)


# Loss Definition


In this section, we will define the loss function used for training the segmentation model. For this tutorial, we will use a combined loss function known as BCEDiceLoss, which is a combination of Binary Cross-Entropy (BCE) Loss and Dice Loss. This combined loss helps to leverage the strengths of both loss functions, improving the performance of the segmentation model.

#### Dice Loss:

Dice loss is commonly used in segmentation models when dealing with imbalanced datasets where the foreground (target object) occupies a much smaller area compared to the background.

#### Leveraging Complementary Strengths of Cross-Entropy Loss:
Cross-entropy loss treats segmentation as a per-pixel classification problem, calculating the loss independently for each pixel. It provides significant gradient updates for pixels that are misclassified with high confidence, helping correct large errors.

In [ ]:
import torch.nn.functional as F

class DiceLoss(nn.Module):
    def __init__(self):
        super(DiceLoss, self).__init__()

    def forward(self, inputs, targets, smooth=1):
        # inputs = torch.sigmoid(inputs)
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()
        dice = (2.*intersection + smooth) / (inputs.sum() + targets.sum() + smooth)
        return 1 - dice


class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super(BCEDiceLoss, self).__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.bce = nn.BCELoss()
        self.dice = DiceLoss()

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        dice_loss = self.dice(inputs, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# Training Step

In this section, we will detail the training step, including the declaration of the optimizer, the loss criteria, and the training procedure for our segmentation model. This step involves initializing the optimizer and the loss function, and then implementing the training loop to update the model's parameters.

In [ ]:
def _step(model,X,Y,criterion, device):

    X = X.to(device)

    if len(Y)>1:
        combined_Ys = []
        for y in Y:
            combined_Ys.append(torch.max(y, dim=0)[0].unsqueeze(0))

        combined_Y = torch.stack(combined_Ys)
    else:
        combined_Y = torch.max(Y[0], dim=0)[0]  # [256, 256]

        combined_Y = combined_Y.unsqueeze(0)


    combined_Y = combined_Y.to(device)

    y_pred = model(X)
    y_pred = y_pred.squeeze(0)



    loss = criterion(y_pred, combined_Y)
    return loss



def train_iters(model, optimizer, num_epochs, train_loader, val_loader, criteria):

    train_losses = []
    val_losses = []

    model = model.to(device)
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        model.train()
        for image, mask, bbox, cls in train_loader:
            optimizer.zero_grad()

            loss = _step(model, image, mask, criteria, device)

            loss.backward()
            optimizer.step()


            epoch_loss += loss.item() * image.size(0)

        total_loss = epoch_loss / len(train_loader)
        train_losses.append(total_loss)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for image, mask, bbox, cls in val_loader:

                loss = _step(model, image, mask, criteria, device)

                val_loss += loss.item() * image.size(0)
                val_losses.append(val_loss)

        val_loss /= len(val_loader.dataset)

        print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {total_loss:.4f}, Val Loss: {val_loss:.4f}')

    return model, train_losses, val_losses

def plot_losses(train_losses, val_losses):
    epochs = range(1, len(train_losses) + 1)
    plt.plot(epochs, train_losses, 'bo-', label='Training loss')
    plt.plot(epochs, val_losses, 'ro-', label='Validation loss')
    plt.title('Training and validation loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()


In [ ]:
num_epochs = 10
optim_1 = Adam(model_1.parameters(), lr = 0.001)
optim_2 = Adam(model_2.parameters(), lr = 0.001)


criteria = BCEDiceLoss(bce_weight=0.5, dice_weight=0.5)
criteria_2 = nn.BCEWithLogitsLoss()

device = 'cuda'

In [ ]:
plot_losses(loss_1, val_1)

In [ ]:
plot_losses(loss_2, val_2)

# Visualizing the results

In [ ]:
def visualize_predictions(model, model_2, dataloader, device):
    model.eval()
    with torch.no_grad():
        for image, mask, bbox, cls in dataloader:
            image = image.to(device)
            prediction = model(image)

            prediction_2 = model_2(image)
            # prediction.to(device)
            # prediction = (prediction > 0.5).float()

            image = image[0].cpu().numpy().squeeze()
            mask = mask[0].cpu().numpy().squeeze()
            prediction = prediction[0].cpu().numpy().squeeze()
            prediction_2 = prediction_2[0].cpu().numpy().squeeze()


            plt.figure(figsize=(12, 4))
            plt.subplot(1, 3, 1)
            plt.imshow(image, cmap='gray')
            plt.title('Input Image')


            plt.subplot(1, 3, 2)
            plt.imshow(prediction, cmap='gray')
            plt.title('Prediction DICE loss')

            plt.subplot(1, 3, 3)
            plt.imshow(prediction_2, cmap='gray')
            plt.title('Prediction BCE loss')

            plt.show()
            break

visualize_predictions(mod_1, mod_2, test_loader, device)